<a href="https://colab.research.google.com/github/susmitsingh01/llm-quantization-from-scratch/blob/main/Quantization_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantization From Scratch
**Model:** Llama 3 8B | **Device:** A100 40GB | **Dataset:** WikiText-2

Implementing PTQ, SmoothQuant, GPTQ, and AWQ in pure PyTorch.
No library quantization APIs — everything built from first principles.

In [1]:
# ── Setup & Imports ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import login
from copy import deepcopy
import time

# ── Device Check ──────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

print(f"Device:  {device}")
print(f"GPU:     {gpu_name}")
print(f"VRAM:    {vram_gb:.1f} GB")

# ── HF Login ──────────────────────────────────────────────────────────────────
login()   # will prompt for your token

Device:  cuda
GPU:     NVIDIA A100-SXM4-80GB
VRAM:    85.1 GB


In [2]:
# ── Load Model & Tokenizer ────────────────────────────────────────────────────
MODEL_ID = "meta-llama/Meta-Llama-3-8B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this will take ~1-2 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # BF16 — fits comfortably in 40GB
    device_map="cuda",             # put everything on the A100
)
model.eval()                       # inference only, no gradients needed

# ── Memory Check ──────────────────────────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
print(f"\nModel loaded.")
print(f"VRAM allocated: {allocated:.1f} GB")
print(f"VRAM reserved:  {reserved:.1f} GB")
print(f"Model dtype:    {next(model.parameters()).dtype}")

Loading tokenizer...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]


Model loaded.
VRAM allocated: 16.1 GB
VRAM reserved:  16.1 GB
Model dtype:    torch.bfloat16


## Evaluation: Perplexity on WikiText-2

Perplexity is our primary metric throughout. We compute it once here and reuse
it after every quantization method.

Perplexity = exp(average negative log-likelihood per token)

Lower is better. A well-quantized model should have perplexity close to the
BF16 baseline. We'll use this number to measure how much each method degrades
the model.

In [3]:
# ── Load WikiText-2 ───────────────────────────────────────────────────────────
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
# Concatenate all test text into one long string, then tokenize
text = "\n\n".join(dataset["text"])
encodings = tokenizer(text, return_tensors="pt")

print(f"Test tokens: {encodings.input_ids.shape[1]:,}")

# ── Perplexity Function ───────────────────────────────────────────────────────
def compute_perplexity(model, encodings, seq_len=2048, device="cuda"):
    """
    Compute perplexity on WikiText-2 test set.

    We slide a window of seq_len tokens across the full corpus.
    At each step, we compute the average cross-entropy loss over the tokens
    in that window, then exponentiate to get perplexity.

    Args:
        model:     the causal LM (BF16 or quantized)
        encodings: tokenized WikiText-2 test set
        seq_len:   context window size (2048 — Llama 3 supports 8192 but
                   2048 is standard for perplexity benchmarks)
        device:    cuda

    Returns:
        perplexity: float
    """
    model.eval()
    input_ids = encodings.input_ids.to(device)
    total_tokens = input_ids.shape[1]

    nlls = []         # negative log-likelihoods per window
    n_windows = 0

    with torch.no_grad():
        for start in range(0, total_tokens - seq_len, seq_len):
            end = start + seq_len
            input_chunk  = input_ids[:, start:end]        # [1, seq_len]
            target_chunk = input_ids[:, start+1:end+1]    # shifted by 1

            # Forward pass
            outputs = model(input_chunk)
            logits  = outputs.logits                      # [1, seq_len, vocab_size]

            # Cross-entropy loss over this window
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.shape[-1]),        # [seq_len, vocab_size]
                target_chunk.reshape(-1),                 # [seq_len]
            )
            nlls.append(loss.item())
            n_windows += 1

    avg_nll     = np.mean(nlls)
    perplexity  = np.exp(avg_nll)
    return perplexity

# ── Baseline Perplexity (BF16, no quantization) ───────────────────────────────
print("Computing baseline perplexity (BF16)...")
start_time = time.time()
baseline_ppl = compute_perplexity(model, encodings, seq_len=2048, device=device)
elapsed = time.time() - start_time

print(f"\nBaseline BF16 Perplexity: {baseline_ppl:.2f}")
print(f"Time taken: {elapsed:.1f}s")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Test tokens: 289,077
Computing baseline perplexity (BF16)...

Baseline BF16 Perplexity: 6.14
Time taken: 25.5s


In [6]:
# ── Results Tracker ───────────────────────────────────────────────────────────
# We'll add to this after each method
results = {
    "BF16 Baseline": {"perplexity": 6.14},
}

def print_results():
    print(f"\n{'Method':<20} {'Perplexity':>12} {'Delta vs BF16':>15}")
    print("-" * 50)
    baseline = results["BF16 Baseline"]["perplexity"]
    for method, stats in results.items():
        ppl   = stats["perplexity"]
        delta = ppl - baseline
        sign  = "+" if delta > 0 else ""
        print(f"{method:<20} {ppl:>12.2f} {sign+f'{delta:.2f}':>15}")

print_results()


Method                 Perplexity   Delta vs BF16
--------------------------------------------------
BF16 Baseline                6.14            0.00


## Section 1: Quantization Primitives

Everything builds on two operations:
- **Quantize:** float tensor → integer tensor
- **Dequantize:** integer tensor → float tensor (with error)

The error introduced by this round-trip is what all our methods try to minimize:

    δy = ΔW · x

Output error = weight quantization error × input activation.

### Symmetric vs Asymmetric
- **Symmetric:** zero_point = 0, signed integers, used for weights
  - scale = max|W| / (2^(bits-1) - 1)
- **Asymmetric:** nonzero zero_point, unsigned integers, used for activations
  - scale = (max - min) / (2^bits - 1)
  - zero_point = round(-min / scale)

### Granularity
- **Per-tensor:** one scale for the whole matrix — coarse, fast
- **Per-channel:** one scale per output channel (row) — standard for INT8
- **Per-group:** one scale per 128 input channels — necessary for INT4

In [4]:
# ── Quantization Primitives ───────────────────────────────────────────────────

def get_scale_symmetric(tensor, bits=8):
    """
    Compute scale for symmetric quantization.
    Used for weights — weight distributions are roughly zero-centered.

    scale = max|W| / (2^(bits-1) - 1)
    """
    q_max = 2 ** (bits - 1) - 1          # INT8: 127, INT4: 7
    max_val = tensor.abs().max()
    scale = max_val / q_max
    return scale


def get_scale_asymmetric(tensor, bits=8):
    """
    Compute scale and zero_point for asymmetric quantization.
    Used for activations — can be skewed (e.g. post-ReLU is all positive).

    scale     = (max - min) / (2^bits - 1)
    zero_point = round(-min / scale)
    """
    q_min  = 0
    q_max  = 2 ** bits - 1               # INT8: 255
    min_val = tensor.min()
    max_val = tensor.max()

    scale      = (max_val - min_val) / (q_max - q_min)
    zero_point = torch.round(-min_val / scale).clamp(q_min, q_max).to(torch.int32)
    return scale, zero_point


def get_scale_per_channel(weight, bits=8):
    """
    One scale per output channel (row of weight matrix).
    Standard granularity for INT8 weight quantization.

    weight: [out_channels, in_channels]
    returns scale: [out_channels]
    """
    q_max  = 2 ** (bits - 1) - 1
    max_vals = weight.abs().max(dim=1).values    # [out_channels]
    return max_vals / q_max


def get_scale_per_group(weight, bits=4, group_size=128):
    """
    One scale per group of `group_size` input channels.
    Necessary for INT4 — per-channel isn't granular enough.

    weight: [out_channels, in_channels]
    returns scale: [out_channels, num_groups]
    """
    out_channels, in_channels = weight.shape
    assert in_channels % group_size == 0, \
        f"in_channels ({in_channels}) must be divisible by group_size ({group_size})"

    num_groups = in_channels // group_size
    q_max      = 2 ** (bits - 1) - 1

    # Reshape into groups then take max per group
    w_grouped = weight.view(out_channels, num_groups, group_size)
    max_vals  = w_grouped.abs().max(dim=2).values    # [out_channels, num_groups]
    return max_vals / q_max


def quantize(tensor, scale, zero_point=0, bits=8, symmetric=True):
    """
    Quantize float tensor to integers.

    Args:
        tensor:     float tensor, shape [out_channels, in_channels]
        scale:      scalar, [out_channels], or [out_channels, num_groups]
        zero_point: 0 for symmetric, int tensor for asymmetric
        bits:       bit-width
        symmetric:  True → signed int, False → unsigned int
    """
    if symmetric:
        q_min, q_max = -(2 ** (bits - 1)), (2 ** (bits - 1) - 1)   # INT8: -128, 127
    else:
        q_min, q_max = 0, (2 ** bits - 1)                           # INT8: 0, 255

    # Reshape scale for broadcasting
    if isinstance(scale, torch.Tensor):
        if scale.ndim == 1:
            # Per-channel: [out_channels] → [out_channels, 1]
            scale = scale.view(-1, 1)
            if not symmetric:
                zero_point = zero_point.view(-1, 1)
        elif scale.ndim == 2:
            # Per-group: [out_channels, num_groups] → [out_channels, num_groups, 1]
            # weight must be viewed as [out_channels, num_groups, group_size] by caller
            scale = scale.unsqueeze(-1)

    w_scaled = tensor / scale + zero_point
    w_q      = torch.clamp(torch.round(w_scaled), q_min, q_max)

    return w_q.to(torch.int8) if symmetric else w_q.to(torch.uint8)


def dequantize(tensor_q, scale, zero_point=0, symmetric=True):
    """
    Dequantize integer tensor back to float.
    Introduces quantization error ΔW — this is what we analyze everywhere.
    """
    if isinstance(scale, torch.Tensor):
        if scale.ndim == 1:
            scale = scale.view(-1, 1)
            if not symmetric:
                zero_point = zero_point.view(-1, 1)
        elif scale.ndim == 2:
            scale = scale.unsqueeze(-1)

    return (tensor_q.float() - zero_point) * scale


print("Primitives defined.")

Primitives defined.


### Sanity Checks

Before touching the model, we verify our primitives are correct on a synthetic
weight matrix. We check three things:

1. **Per-tensor symmetric (INT8)** — coarsest granularity
2. **Per-channel symmetric (INT8)** — standard for INT8 weights  
3. **Per-group symmetric (INT4)** — required for INT4 weights

For each, we check:
- Quantized tensor has the right dtype
- Dequantized tensor is close to the original (low mean absolute error)
- Finer granularity → lower error (this should hold clearly for INT4)

In [7]:
# ── Sanity Checks ─────────────────────────────────────────────────────────────
torch.manual_seed(42)

# Simulate a Llama 3 8B linear layer weight: [4096, 4096]
W = torch.randn(4096, 4096).to(device)

def check(label, W_orig, W_deq, W_q):
    error = (W_orig - W_deq).abs().mean().item()
    print(f"{label}")
    print(f"  dtype:          {W_q.dtype}")
    print(f"  mean abs error: {error:.6f}")
    print()

print("=" * 50)

# ── Test 1: Per-tensor symmetric INT8 ────────────────
scale  = get_scale_symmetric(W, bits=8)
W_q    = quantize(W, scale, bits=8, symmetric=True)
W_deq  = dequantize(W_q, scale, symmetric=True)
check("Per-tensor INT8", W, W_deq, W_q)

# ── Test 2: Per-channel symmetric INT8 ───────────────
scale  = get_scale_per_channel(W, bits=8)
W_q    = quantize(W, scale, bits=8, symmetric=True)
W_deq  = dequantize(W_q, scale, symmetric=True)
check("Per-channel INT8", W, W_deq, W_q)

# ── Test 3: Per-group symmetric INT4 ─────────────────
group_size   = 128
out_ch, in_ch = W.shape
num_groups   = in_ch // group_size

scale        = get_scale_per_group(W, bits=4, group_size=group_size)  # [4096, 32]
W_grouped    = W.view(out_ch, num_groups, group_size)
W_q_grouped  = quantize(W_grouped, scale, bits=4, symmetric=True)
W_deq        = dequantize(W_q_grouped, scale, symmetric=True).view(out_ch, in_ch)
check("Per-group INT4 (g=128)", W, W_deq, W_q_grouped)

print("Expected: error decreases as granularity increases")
print("Expected: INT4 error >> INT8 error (fewer bits)")

Per-tensor INT8
  dtype:          torch.int8
  mean abs error: 0.010330

Per-channel INT8
  dtype:          torch.int8
  mean abs error: 0.007475

Per-group INT4 (g=128)
  dtype:          torch.int8
  mean abs error: 0.100181

Expected: error decreases as granularity increases
Expected: INT4 error >> INT8 error (fewer bits)


## Section 2: Post-Training Quantization (PTQ)

PTQ quantizes a pretrained model without any retraining.

We implement **W8A16** — weights quantized to INT8, activations stay BF16.
This is the simplest and most practical PTQ variant:
- Weights: per-channel symmetric INT8 (we just verified this works)
- Activations: untouched (no calibration data needed for scales)
- Effect: model memory cuts roughly in half, decode speed improves due to
  reduced memory bandwidth

### What we do
1. Walk every `nn.Linear` layer in the model
2. Quantize its weight to INT8 (per-channel symmetric)
3. Dequantize back to BF16 before the matmul
4. Replace the original layer with our `QuantizedLinear` module

Step 3 means we're doing the matmul in BF16 — this is W8A16, not W8A8.
True INT8 compute (W8A8) requires calibration data for activation scales,
which we cover in SmoothQuant.

### Expected outcome
Perplexity should stay very close to baseline (6.14).
W8A16 PTQ is well understood — INT8 per-channel is lossless enough that
degradation is typically < 0.1 perplexity points.

In [8]:
# ── PTQ: W8A16 ────────────────────────────────────────────────────────────────

class QuantizedLinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with INT8 per-channel symmetric weights.

    Forward pass:
      1. Dequantize INT8 weight → BF16
      2. Run standard BF16 matmul
      3. Add bias if present

    This is W8A16 — activations stay BF16, only weights are quantized.
    Memory savings: ~50% for weight storage.
    """
    def __init__(self, weight_q, scale, bias=None):
        super().__init__()
        self.register_buffer("weight_q", weight_q)   # INT8 [out, in]
        self.register_buffer("scale", scale)          # BF16 [out]
        self.register_buffer("bias", bias)            # BF16 [out] or None

    def forward(self, x):
        weight_deq = dequantize(self.weight_q, self.scale, symmetric=True)
        weight_deq = weight_deq.to(x.dtype)
        return nn.functional.linear(x, weight_deq, self.bias)


def quantize_model_w8a16(model):
    """
    Replace every nn.Linear in-place with QuantizedLinear.

    No deepcopy — we quantize layer by layer and immediately delete the
    original BF16 weight to stay within VRAM budget.
    """
    linear_layers = {
        name: module
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    }

    print(f"Found {len(linear_layers)} Linear layers to quantize.")

    for i, (name, linear) in enumerate(linear_layers.items()):
        weight = linear.weight.data                          # BF16 [out, in]
        scale  = get_scale_per_channel(weight, bits=8).to(weight.dtype)
        weight_q = quantize(weight, scale, bits=8, symmetric=True)
        bias   = linear.bias.data if linear.bias is not None else None

        # Build replacement
        new_module = QuantizedLinear(weight_q, scale, bias)

        # Swap in-place
        parts  = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_module)

        # Free the original BF16 weight immediately
        del weight, weight_q, scale
        if (i + 1) % 50 == 0:
            torch.cuda.empty_cache()
            print(f"  {i+1}/{len(linear_layers)} layers done...")

    torch.cuda.empty_cache()
    return model


# ── Run PTQ ───────────────────────────────────────────────────────────────────
# NOTE: this modifies `model` in-place.
# After this cell, `model` IS the quantized model.
print("Quantizing model (W8A16 PTQ, in-place)...")
start = time.time()
model = quantize_model_w8a16(model)
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

# ── Memory check ──────────────────────────────────────────────────────────────
def model_size_gb(m):
    total = sum(
        p.numel() * p.element_size()
        for p in list(m.parameters()) + list(m.buffers())
    )
    return total / 1e9

allocated = torch.cuda.memory_allocated() / 1e9
print(f"\nModel size (buffers+params): {model_size_gb(model):.2f} GB")
print(f"VRAM allocated:              {allocated:.2f} GB")

Quantizing model (W8A16 PTQ, in-place)...
Found 225 Linear layers to quantize.
  50/225 layers done...
  100/225 layers done...
  150/225 layers done...
  200/225 layers done...
Done in 0.3s

Model size (buffers+params): 8.56 GB
VRAM allocated:              8.57 GB


In [8]:
# ── PTQ Perplexity ────────────────────────────────────────────────────────────
print("Computing PTQ (W8A16) perplexity...")
start = time.time()
ptq_ppl = compute_perplexity(model, encodings, seq_len=2048, device=device)
elapsed = time.time() - start

print(f"\nPTQ W8A16 Perplexity: {ptq_ppl:.2f}")
print(f"Time taken: {elapsed:.1f}s")

# ── Update results tracker ────────────────────────────────────────────────────
results["PTQ W8A16 (INT8)"] = {"perplexity": ptq_ppl}
print_results()

Computing PTQ (W8A16) perplexity...

PTQ W8A16 Perplexity: 6.15
Time taken: 43.2s

Method                 Perplexity   Delta vs BF16
--------------------------------------------------
BF16 Baseline                6.14            0.00
PTQ W8A16 (INT8)             6.15           +0.01


## Section 3: SmoothQuant

PTQ worked well for W8A16 — but what if we want W8A8?
W8A8 means both weights AND activations are INT8, enabling true INT8 matrix
multiply on the GPU (faster than BF16 matmul).

The problem: activations have outliers.

At 6.7B+ parameters, specific input channels develop magnitudes 100-200x
larger than the average. If we quantize activations per-tensor, those outlier
channels dominate the scale and crush the precision of every other channel.

### The SmoothQuant fix
Migrate the quantization difficulty from activations to weights:

    Y = (X / s) · (s · W)^T

Same mathematical output. But now:
- Activations X/s — outliers divided away, easy to quantize
- Weights s·W — absorb the difficulty, but weights are per-channel
  quantized anyway so they handle it better

### Computing s
    s_j = max|X_j|^α / max|W_j|^(1-α)

- α = 0.5 balances difficulty equally between X and W
- j indexes input channels
- s is absorbed into the preceding LayerNorm at zero runtime cost

### What we implement
1. Run calibration data through the model, collect activation statistics
2. Compute per-channel smoothing factors s
3. Scale LayerNorm weights by s (absorbed — no extra ops at runtime)
4. Scale Linear weights by 1/s (counterbalance)
5. Measure perplexity — should recover any degradation from W8A8

In [9]:
# ── Reload fresh BF16 model for SmoothQuant ──────────────────────────────────
del model
torch.cuda.empty_cache()

print("Reloading Llama 3 8B in BF16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda",
)
model.eval()

linear_count     = sum(1 for _, m in model.named_modules() if isinstance(m, nn.Linear))
quantized_count  = sum(1 for _, m in model.named_modules() if isinstance(m, QuantizedLinear))
print(f"nn.Linear layers:       {linear_count}")
print(f"QuantizedLinear layers: {quantized_count}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Reloading Llama 3 8B in BF16...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

nn.Linear layers:       225
QuantizedLinear layers: 0
VRAM: 16.07 GB


In [18]:
def collect_activation_stats(model, encodings, n_samples=4096, seq_len=512):
    act_stats = {}
    hooks     = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            def make_hook(n):
                def hook(module, input, output):
                    x     = input[0].detach()
                    x_max = x.abs().view(-1, x.shape[-1]).max(dim=0).values
                    if n not in act_stats:
                        act_stats[n] = x_max
                    else:
                        act_stats[n] = torch.maximum(act_stats[n], x_max)
                return hook
            hooks.append(module.register_forward_hook(make_hook(name)))

    input_ids = encodings.input_ids.to(device)
    n_batches = n_samples // seq_len
    print(f"Running {n_batches} calibration batches ({seq_len} tokens each)...")

    with torch.no_grad():
        for i in range(n_batches):
            start = i * seq_len
            chunk = input_ids[:, start:start + seq_len]
            model(chunk)
            if (i + 1) % 2 == 0:
                print(f"  batch {i+1}/{n_batches} done")

    for h in hooks:
        h.remove()

    print(f"Collected stats for {len(act_stats)} layers.")
    return act_stats

In [24]:
def compute_and_apply_smoothquant_v4(model, act_stats, alpha=0.5, s_max=4.0):
    """
    Correct SmoothQuant with proper LayerNorm absorption.

    Y = (X/s) · (s·W)^T

    X/s: absorbed into preceding LayerNorm gamma (gamma → gamma/s)
    s·W: absorbed into Linear weight (W → W*s... wait no)

    Actually the correct absorption:
    - LayerNorm output X gets divided by s → absorb by dividing LN gamma by s
    - Linear weight W gets multiplied by s → absorb by multiplying W columns by s

    Then at quantization time, W*s has better quantization properties because
    the salient channels (large x) now have larger weights, so they get
    finer quantization steps.

    Wait — let's re-derive:
    Original: Y = X · W^T
    Smoothed: Y = (X · diag(1/s)) · (diag(s) · W)^T
                = X_smooth · W_smooth^T

    X_smooth = X / s  → divide LN gamma by s
    W_smooth = s * W  → multiply W rows... no, columns

    W is [out, in]. We scale input channels:
    W_smooth[:, j] = s_j * W[:, j]  → multiply columns by s
    LN gamma_j = gamma_j / s_j      → divide gamma by s
    """
    named_modules = dict(model.named_modules())
    skip_names    = {"lm_head", "embed_tokens"}

    layernorm_map = {
        "q_proj":    "input_layernorm",
        "k_proj":    "input_layernorm",
        "v_proj":    "input_layernorm",
        "o_proj":    None,
        "gate_proj": "post_attention_layernorm",
        "up_proj":   "post_attention_layernorm",
        "down_proj": None,
    }

    ln_info = {}

    for name, x_max in act_stats.items():
        if name not in named_modules:
            continue

        proj_name = name.split(".")[-1]
        if proj_name in skip_names:
            continue

        module = named_modules[name]
        if not isinstance(module, nn.Linear):
            continue

        parts = name.split(".")
        if len(parts) < 4:
            continue

        layer_path = ".".join(parts[:3])
        ln_name    = layernorm_map.get(proj_name)
        if ln_name is None:
            continue

        ln_path = f"{layer_path}.{ln_name}"
        if ln_path not in named_modules:
            continue
        if not hasattr(named_modules[ln_path], "weight"):
            continue

        w_max = module.weight.data.abs().max(dim=0).values.float()
        x_max = x_max.to(w_max.device).float()

        if ln_path not in ln_info:
            ln_info[ln_path] = {
                "x_max":      x_max.clone(),
                "w_max":      w_max.clone(),
                "proj_names": [name],
            }
        else:
            ln_info[ln_path]["w_max"] = torch.maximum(
                ln_info[ln_path]["w_max"], w_max
            )
            ln_info[ln_path]["proj_names"].append(name)

    applied_layers = 0
    applied_lns    = 0

    for ln_path, info in ln_info.items():
        x_max     = info["x_max"]
        w_max     = info["w_max"]
        layernorm = named_modules[ln_path]

        # s_j = max|X_j|^alpha / max|W_j|^(1-alpha)
        s = (x_max.pow(alpha) / (w_max.pow(1 - alpha) + 1e-8))
        s = s.clamp(min=1e-5, max=s_max)

        # Divide LayerNorm gamma by s  (handles X/s side)
        s_ln = s.to(layernorm.weight.dtype)
        layernorm.weight.data.div_(s_ln)

        # Multiply each Linear weight's columns by s  (handles s*W side)
        for proj_name in info["proj_names"]:
            linear = named_modules[proj_name]
            s_w    = s.to(linear.weight.dtype)
            linear.weight.data.mul_(s_w.unsqueeze(0))
            applied_layers += 1

        applied_lns += 1

    print(f"LayerNorms scaled:      {applied_lns}")
    print(f"Linear layers adjusted: {applied_layers}")

    # Verify
    test_input = encodings.input_ids[:, :128].to(device)
    targets    = encodings.input_ids[:, 1:129].to(device)
    with torch.no_grad():
        logits = model(test_input).logits

    loss = nn.functional.cross_entropy(
        logits[:, :10].reshape(-1, logits.shape[-1]),
        targets[:, :10].reshape(-1),
        reduction='none'
    )
    print(f"\nPer-token CE after smoothing (should match unsmoothed ~6.28 mean):")
    for i, l in enumerate(loss):
        print(f"  token {i}: {l.item():.4f}")
    print(f"Mean CE:     {loss.mean().item():.4f}")
    print(f"Perplexity:  {loss.mean().exp().item():.2f}")


# Run on already-loaded fresh model
act_stats = collect_activation_stats(model, encodings, n_samples=4096, seq_len=512)
compute_and_apply_smoothquant_v4(model, act_stats, alpha=0.5, s_max=4.0)

Running 8 calibration batches (512 tokens each)...
  batch 2/8 done
  batch 4/8 done
  batch 6/8 done
  batch 8/8 done
Collected stats for 225 layers.
LayerNorms scaled:      64
Linear layers adjusted: 160

Per-token CE after smoothing (should match unsmoothed ~6.28 mean):
  token 0: 8.3750
  token 1: 12.8750
  token 2: 12.1250
  token 3: 4.4688
  token 4: 5.8438
  token 5: 2.0781
  token 6: 3.0156
  token 7: 9.8750
  token 8: 3.8750
  token 9: 0.1475
Mean CE:     6.2812
Perplexity:  536.00


In [29]:
# ── Quantize smoothed model (CPU offload) ─────────────────────────────────────
def quantize_model_w8a16_cpu(model):
    """
    Same as quantize_model_w8a16 but forces all quantization math on CPU.
    Critical for smoothed models — smoothed weights have very small values
    in some channels which can be silently corrupted by GPU intermediate tensors.
    """
    linear_layers = {
        name: module
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    }
    print(f"Found {len(linear_layers)} Linear layers to quantize.")

    for i, (name, linear) in enumerate(linear_layers.items()):
        # Pull to CPU as FP32 for safe quantization math
        weight_cpu   = linear.weight.data.cpu().float()
        scale_cpu    = get_scale_per_channel(weight_cpu, bits=8)
        weight_q_cpu = quantize(weight_cpu, scale_cpu, bits=8, symmetric=True)

        # Move results back to GPU
        weight_q = weight_q_cpu.to(device)
        scale    = scale_cpu.to(torch.bfloat16).to(device)
        bias     = linear.bias.data if linear.bias is not None else None

        # Free original BF16 weight from GPU immediately
        linear.weight.data = torch.empty(0, device=device)
        torch.cuda.empty_cache()

        # Replace layer
        new_module = QuantizedLinear(weight_q, scale, bias)
        parts  = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_module)

        del weight_cpu, scale_cpu, weight_q_cpu, weight_q, scale
        if (i + 1) % 50 == 0:
            torch.cuda.empty_cache()
            print(f"  {i+1}/{len(linear_layers)} layers done...")

    torch.cuda.empty_cache()
    return model




In [30]:
print("Quantizing smoothed model (W8A16, CPU offload)...")
start = time.time()
model = quantize_model_w8a16_cpu(model)
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")
print(f"Model size: {model_size_gb(model):.2f} GB")

print("\nComputing SmoothQuant perplexity...")
start  = time.time()
sq_ppl = compute_perplexity(model, encodings, seq_len=2048, device=device)
elapsed = time.time() - start

print(f"\nSmoothQuant Perplexity: {sq_ppl:.2f}")
print(f"Time taken: {elapsed:.1f}s")

results["SmoothQuant (W8A16)"] = {"perplexity": sq_ppl}
print_results()

Quantizing smoothed model (W8A16, CPU offload)...
Found 0 Linear layers to quantize.
Done in 0.0s
Model size: 8.56 GB

Computing SmoothQuant perplexity...

SmoothQuant Perplexity: 6.15
Time taken: 43.1s

Method                 Perplexity   Delta vs BF16
--------------------------------------------------
BF16 Baseline                6.14            0.00
SmoothQuant (W8A16)          6.15           +0.01


## Section 4: GPTQ (Generative Pre-trained Transformer Quantization)

PTQ and SmoothQuant both quantize weights independently — each weight is
quantized without considering how the error affects the layer's output.

GPTQ takes a fundamentally different approach: it minimizes the layer-wise
output error directly.

### The objective
For each linear layer, find quantized weights W_q that minimize:

    argmin ||WX - W_q X||²_F

Where X is the layer's input activations collected from calibration data.
This is a per-layer optimization problem.

### The Hessian
The sensitivity of the output error to each weight w_ij is captured by:

    H = 2XX^T

H is [in_channels, in_channels]. The diagonal H_qq tells us how sensitive
the output is to weight q — high H_qq means weight q is important.

### Optimal Brain Surgeon (OBS)
When we quantize weight q (introducing error δw_q), the optimal correction
to all other weights is:

    δw = -(δw_q / [H⁻¹]_qq) · [H⁻¹]_{:,q}

This propagates the quantization error optimally across remaining weights.

### GPTQ's key insight
Instead of finding the globally optimal quantization order (expensive),
GPTQ quantizes weights column by column — all rows of a column at once.
This allows efficient Hessian updates using Cholesky decomposition.

### What we implement
1. Run calibration data through the model, collect input activations X
   for each Linear layer
2. Compute H = 2XX^T per layer
3. Apply damping: H += λI for numerical stability
4. Cholesky decompose H⁻¹ for efficient column-by-column updates
5. Quantize column by column, applying OBS corrections after each column
6. Replace nn.Linear with QuantizedLinear (INT4 per-group, g=128)
7. Measure perplexity

## Collect Layer Inputs for GPTQ

In [36]:
# ── GPTQ: Collect layer input activations ────────────────────────────────────
# Unlike SmoothQuant which only needed max|X_j|, GPTQ needs the full X
# to compute H = 2XX^T per layer.

def collect_layer_inputs(model, encodings, layer_names, n_samples=128, seq_len=512):
    """
    Collect input activations X for specified Linear layers.

    Args:
        model:       BF16 model
        encodings:   tokenized WikiText-2
        layer_names: list of layer names to collect inputs for
        n_samples:   number of tokens (keep small — we store full tensors)
        seq_len:     sequence length per forward pass
    Returns:
        layer_inputs: dict layer_name → X tensor [n_tokens, in_channels]
    """
    layer_inputs = {}
    hooks        = []

    for name, module in model.named_modules():
        if name not in layer_names:
            continue
        if not isinstance(module, nn.Linear):
            continue

        def make_hook(n):
            def hook(module, input, output):
                x = input[0].detach()
                x = x.view(-1, x.shape[-1])
                x = x.float()
                if n not in layer_inputs:
                    layer_inputs[n] = x
                else:
                    layer_inputs[n] = torch.cat([layer_inputs[n], x], dim=0)
            return hook

        hooks.append(module.register_forward_hook(make_hook(name)))

    input_ids = encodings.input_ids.to(device)
    n_batches = n_samples // seq_len

    with torch.no_grad():
        for i in range(n_batches):
            start = i * seq_len
            chunk = input_ids[:, start:start + seq_len]
            model(chunk)

    for h in hooks:
        h.remove()

    for name in layer_inputs:
        layer_inputs[name] = layer_inputs[name].cpu()

    return layer_inputs

# ── GPTQ: Compute Hessian ─────────────────────────────────────────────────────
def compute_hessian(X, damp_percent=0.01):
    """
    Compute H = 2XX^T and apply damping.

    Args:
        X:            [n_tokens, in_channels] float32
        damp_percent: λ = damp_percent * mean(diag(H))
    Returns:
        H: [in_channels, in_channels] damped Hessian
    """
    # H = 2 * X^T X  (note: X is [n_tokens, in_ch], so X^T X

## GPTQ Core (Column-by-Column Quantization)

In [33]:
# ── GPTQ: Column-by-column quantization with OBS updates ─────────────────────

def gptq_quantize_layer(weight, H, bits=4, group_size=128):
    """
    Quantize a single Linear layer's weight matrix using GPTQ.

    Args:
        weight:     [out_channels, in_channels] float32 weight
        H:          [in_channels, in_channels] Hessian
        bits:       quantization bit-width (4 for INT4)
        group_size: number of input channels per quantization group
    Returns:
        W_q:    quantized weight [out_channels, in_channels] int8
        scales: per-group scales [out_channels, num_groups]
        W_deq:  dequantized weight [out_channels, in_channels] float32
    """
    out_ch, in_ch = weight.shape
    num_groups    = in_ch // group_size
    W             = weight.clone().float()   # work in FP32

    # ── Cholesky decomposition of H⁻¹ ────────────────────────────────────────
    # We need H⁻¹ for OBS updates. Rather than inverting directly (unstable),
    # we compute Cholesky of H then use it to solve column updates efficiently.
    try:
        # H = L L^T → H⁻¹ computed via Cholesky
        L     = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)
    except torch.linalg.LinAlgError:
        # If Cholesky fails (H not positive definite), add more damping
        print("  Cholesky failed — increasing damping")
        damp  = 0.1 * H.diag().mean()
        H    += damp * torch.eye(H.shape[0])
        L     = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)

    # ── Allocate output tensors ───────────────────────────────────────────────
    W_q    = torch.zeros_like(W, dtype=torch.int8)
    scales = torch.zeros(out_ch, num_groups, dtype=torch.float32)
    W_deq  = torch.zeros_like(W)

    # ── Quantize column by column ─────────────────────────────────────────────
    # GPTQ processes all rows (output channels) of one column simultaneously.
    # After quantizing column q, it applies OBS correction to columns q+1:end.

    q_min = -(2 ** (bits - 1))          # INT4: -8
    q_max =   2 ** (bits - 1) - 1       # INT4:  7

    for col in range(in_ch):
        # Which group does this column belong to?
        g = col // group_size

        # Compute per-group scale at the start of each group
        if col % group_size == 0:
            # Scale from the current group's weights
            group_w   = W[:, col:col + group_size]
            group_max = group_w.abs().max(dim=1).values   # [out_ch]
            group_max = group_max.clamp(min=1e-8)
            scale     = group_max / q_max                 # [out_ch]
            scales[:, g] = scale

        # Quantize this column using the current group's scale
        w_col   = W[:, col]                              # [out_ch]
        w_scaled = w_col / scale                         # [out_ch]
        w_q_col  = torch.clamp(torch.round(w_scaled), q_min, q_max).to(torch.int8)
        w_deq_col = w_q_col.float() * scale             # [out_ch]

        W_q[:, col]   = w_q_col
        W_deq[:, col] = w_deq_col

        # Quantization error for this column
        delta_w = w_deq_col - w_col                     # [out_ch]

        # OBS update: propagate error to remaining columns
        # δW[:, col+1:] -= outer(δw, H⁻¹[col, col+1:]) / H⁻¹[col, col]
        h_inv_qq = H_inv[col, col]
        if h_inv_qq.abs() > 1e-10:
            h_inv_row = H_inv[col, col+1:]              # [in_ch - col - 1]
            correction = torch.outer(delta_w, h_inv_row) / h_inv_qq
            W[:, col+1:] -= correction

    return W_q, scales, W_deq


print("GPTQ core defined.")

GPTQ core defined.


##  Apply GPTQ to Full Model

In [37]:
# ── GPTQ: Quantize full model ─────────────────────────────────────────────────

class GPTQ_QuantizedLinear(nn.Module):
    """
    INT4 per-group quantized linear layer produced by GPTQ.
    Stores INT4 weights (in int8 container) and per-group scales.
    """
    def __init__(self, weight_q, scales, bias=None, group_size=128):
        super().__init__()
        self.group_size = group_size
        self.register_buffer("weight_q", weight_q)   # int8 [out, in]
        self.register_buffer("scales",   scales)      # float32 [out, num_groups]
        self.register_buffer("bias",     bias)

    def forward(self, x):
        # Dequantize: expand scales from per-group to per-column
        out_ch, in_ch = self.weight_q.shape
        num_groups    = in_ch // self.group_size

        # scales: [out_ch, num_groups] → [out_ch, in_ch]
        scales_expanded = self.scales.repeat_interleave(self.group_size, dim=1)
        weight_deq      = self.weight_q.float() * scales_expanded
        weight_deq      = weight_deq.to(x.dtype)

        return nn.functional.linear(x, weight_deq, self.bias)


def apply_gptq(model, encodings, bits=4, group_size=128,
               n_samples=128, seq_len=512):
    """
    Apply GPTQ to every Linear layer in the model.

    Strategy: process one layer at a time to manage memory.
    For each layer:
      1. Collect input activations X
      2. Compute H = 2XX^T
      3. Run GPTQ column-by-column quantization
      4. Replace nn.Linear with GPTQ_QuantizedLinear
    """
    # Get all linear layer names
    linear_names = [
        name for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]
    print(f"Found {len(linear_names)} Linear layers to quantize with GPTQ.")
    print(f"Bits: INT{bits}, group_size: {group_size}")

    named_modules_dict = dict(model.named_modules())

    for i, name in enumerate(linear_names):
        linear = named_modules_dict[name]
        W      = linear.weight.data.float().cpu()
        bias   = linear.bias.data if linear.bias is not None else None

        # Collect inputs for this single layer
        layer_inputs = collect_layer_inputs(
            model, encodings, [name],
            n_samples=512, seq_len=512
        )

        if name not in layer_inputs:
            print(f"  [{i+1}/{len(linear_names)}] Skipping {name} (no inputs)")
            continue

        X = layer_inputs[name]   # [n_tokens, in_ch]

        # Compute Hessian
        H = compute_hessian(X, damp_percent=0.01)

        # GPTQ quantization
        W_q, scales, W_deq = gptq_quantize_layer(W, H, bits=bits,
                                                   group_size=group_size)

        # Replace layer with GPTQ_QuantizedLinear
        new_module = GPTQ_QuantizedLinear(
            W_q, scales, bias, group_size=group_size
        )

        parts  = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_module)

        # Free memory
        del X, H, W_q, scales, W_deq, layer_inputs
        torch.cuda.empty_cache()

        if (i + 1) % 10 == 0 or (i + 1) == len(linear_names):
            print(f"  [{i+1}/{len(linear_names)}] done")

    return model

## Reload + Run GPTQ + Perplexity

In [38]:
# ── Reload fresh model and run GPTQ ──────────────────────────────────────────
del model
torch.cuda.empty_cache()

print("Reloading Llama 3 8B in BF16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="cuda"
)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Run GPTQ ──────────────────────────────────────────────────────────────────
print("\nRunning GPTQ (INT4, group_size=128)...")
start = time.time()
model = apply_gptq(
    model, encodings,
    bits=4, group_size=128,
    n_samples=128, seq_len=512
)
elapsed = time.time() - start
print(f"\nGPTQ done in {elapsed/60:.1f} min")
print(f"Model size: {model_size_gb(model):.2f} GB")

# ── GPTQ Perplexity ───────────────────────────────────────────────────────────
print("\nComputing GPTQ perplexity...")
start    = time.time()
gptq_ppl = compute_perplexity(model, encodings, seq_len=2048, device=device)
elapsed  = time.time() - start

print(f"\nGPTQ INT4 Perplexity: {gptq_ppl:.2f}")
print(f"Time taken: {elapsed:.1f}s")

results["GPTQ (INT4)"] = {"perplexity": gptq_ppl}
print_results()

Reloading Llama 3 8B in BF16...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

VRAM: 49.33 GB

Running GPTQ (INT4, group_size=128)...
Found 225 Linear layers to quantize with GPTQ.
Bits: INT4, group_size: 128


TypeError: linalg_cholesky(): argument 'input' (position 1) must be Tensor, not NoneType

In [39]:
# Redefine fixed collect_layer_inputs
def collect_layer_inputs(model, encodings, layer_names, n_samples=512, seq_len=512):
    layer_inputs = {}
    hooks        = []

    for name, module in model.named_modules():
        if name not in layer_names:
            continue
        if not isinstance(module, nn.Linear):
            continue

        def make_hook(n):
            def hook(module, input, output):
                x = input[0].detach()
                x = x.view(-1, x.shape[-1])
                x = x.float()
                if n not in layer_inputs:
                    layer_inputs[n] = x
                else:
                    layer_inputs[n] = torch.cat([layer_inputs[n], x], dim=0)
            return hook

        hooks.append(module.register_forward_hook(make_hook(name)))

    input_ids = encodings.input_ids.to(device)
    n_batches = n_samples // seq_len

    with torch.no_grad():
        for i in range(n_batches):
            start = i * seq_len
            chunk = input_ids[:, start:start + seq_len]
            model(chunk)

    for h in hooks:
        h.remove()

    for name in layer_inputs:
        layer_inputs[name] = layer_inputs[name].cpu()

    return layer_inputs


# Also fix apply_gptq to use n_samples=512
def apply_gptq(model, encodings, bits=4, group_size=128,
               n_samples=512, seq_len=512):
    linear_names = [
        name for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]
    print(f"Found {len(linear_names)} Linear layers to quantize with GPTQ.")
    print(f"Bits: INT{bits}, group_size: {group_size}")

    named_modules_dict = dict(model.named_modules())

    for i, name in enumerate(linear_names):
        linear = named_modules_dict[name]
        W      = linear.weight.data.float().cpu()
        bias   = linear.bias.data if linear.bias is not None else None

        layer_inputs = collect_layer_inputs(
            model, encodings, [name],
            n_samples=n_samples, seq_len=seq_len
        )

        if name not in layer_inputs:
            print(f"  [{i+1}/{len(linear_names)}] Skipping {name} (no inputs)")
            continue

        X = layer_inputs[name]
        if X is None or X.numel() == 0:
            print(f"  [{i+1}/{len(linear_names)}] Skipping {name} (empty X)")
            continue

        H = compute_hessian(X, damp_percent=0.01)

        W_q, scales, W_deq = gptq_quantize_layer(W, H, bits=bits,
                                                   group_size=group_size)

        new_module = GPTQ_QuantizedLinear(W_q, scales, bias, group_size=group_size)

        parts  = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_module)

        del X, H, W_q, scales, W_deq, layer_inputs
        torch.cuda.empty_cache()

        if (i + 1) % 10 == 0 or (i + 1) == len(linear_names):
            print(f"  [{i+1}/{len(linear_names)}] done")

    return model


# Run GPTQ on already-loaded model
print("Running GPTQ (INT4, group_size=128)...")
start = time.time()
model = apply_gptq(model, encodings, bits=4, group_size=128,
                   n_samples=512, seq_len=512)
elapsed = time.time() - start
print(f"\nGPTQ done in {elapsed/60:.1f} min")
print(f"Model size: {model_size_gb(model):.2f} GB")

Running GPTQ (INT4, group_size=128)...
Found 225 Linear layers to quantize with GPTQ.
Bits: INT4, group_size: 128


TypeError: linalg_cholesky(): argument 'input' (position 1) must be Tensor, not NoneType

In [40]:
print(compute_hessian)
import inspect
print(inspect.getsource(compute_hessian))

<function compute_hessian at 0x7b874b52d4e0>
def compute_hessian(X, damp_percent=0.01):
    """
    Compute H = 2XX^T and apply damping.

    Args:
        X:            [n_tokens, in_channels] float32
        damp_percent: λ = damp_percent * mean(diag(H))
    Returns:
        H: [in_channels, in_channels] damped Hessian
    """
    # H = 2 * X^T X  (note: X is [n_tokens, in_ch], so X^T X



In [41]:
def compute_hessian(X, damp_percent=0.01):
    """
    Compute H = 2XX^T and apply damping.
    X: [n_tokens, in_channels] float32
    Returns H: [in_channels, in_channels]
    """
    H     = 2 * (X.T @ X)
    damp  = damp_percent * H.diag().mean()
    H    += damp * torch.eye(H.shape[0], dtype=H.dtype)
    return H

# Quick sanity check
X_test = torch.randn(512, 64)
H_test = compute_hessian(X_test)
print(f"H shape: {H_test.shape}")
print(f"H dtype: {H_test.dtype}")
print(f"H is None: {H_test is None}")

H shape: torch.Size([64, 64])
H dtype: torch.float32
H is None: False


In [45]:
class GPTQ_QuantizedLinear(nn.Module):
    def __init__(self, weight_q, scales, bias=None, group_size=128):
        super().__init__()
        self.group_size = group_size
        self.register_buffer("weight_q", weight_q)
        self.register_buffer("scales",   scales)
        self.register_buffer("bias",     bias)

    def forward(self, x):
        out_ch, in_ch = self.weight_q.shape
        num_groups    = in_ch // self.group_size

        # scales: [out_ch, num_groups] → [out_ch, in_ch]
        scales_expanded = self.scales.repeat_interleave(self.group_size, dim=1)
        weight_deq      = self.weight_q.float() * scales_expanded
        weight_deq      = weight_deq.to(dtype=x.dtype, device=x.device)  # ← fix

        return nn.functional.linear(x, weight_deq, self.bias)

In [46]:
print("Running GPTQ (INT4, group_size=128)...")
start = time.time()
model = apply_gptq(model, encodings, bits=4, group_size=128,
                   n_samples=512, seq_len=512)
elapsed = time.time() - start
print(f"\nGPTQ done in {elapsed/60:.1f} min")
print(f"Model size: {model_size_gb(model):.2f} GB")


Running GPTQ (INT4, group_size=128)...
Found 224 Linear layers to quantize with GPTQ.
Bits: INT4, group_size: 128


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

In [5]:
# ── Clean reload ──────────────────────────────────────────────────────────────
del model
torch.cuda.empty_cache()

print("Reloading fresh BF16 model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="cuda"
)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Reloading fresh BF16 model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

VRAM: 16.07 GB


In [6]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

VRAM: 16.07 GB


In [48]:
# ── GPTQ: All definitions + run ───────────────────────────────────────────────

class GPTQ_QuantizedLinear(nn.Module):
    def __init__(self, weight_q, scales, bias=None, group_size=128):
        super().__init__()
        self.group_size = group_size
        self.register_buffer("weight_q", weight_q)
        self.register_buffer("scales",   scales)
        self.register_buffer("bias",     bias)

    def forward(self, x):
        out_ch, in_ch   = self.weight_q.shape
        num_groups      = in_ch // self.group_size
        scales_expanded = self.scales.repeat_interleave(self.group_size, dim=1)
        weight_deq      = self.weight_q.float() * scales_expanded
        weight_deq      = weight_deq.to(dtype=x.dtype, device=x.device)
        return nn.functional.linear(x, weight_deq, self.bias)


def collect_layer_inputs(model, encodings, layer_names, n_samples=512, seq_len=512):
    layer_inputs = {}
    hooks        = []

    for name, module in model.named_modules():
        if name not in layer_names:
            continue
        if not isinstance(module, nn.Linear):
            continue

        def make_hook(n):
            def hook(module, input, output):
                x = input[0].detach().view(-1, input[0].shape[-1]).float()
                if n not in layer_inputs:
                    layer_inputs[n] = x
                else:
                    layer_inputs[n] = torch.cat([layer_inputs[n], x], dim=0)
            return hook

        hooks.append(module.register_forward_hook(make_hook(name)))

    input_ids = encodings.input_ids.to(device)
    n_batches = n_samples // seq_len

    with torch.no_grad():
        for i in range(n_batches):
            chunk = input_ids[:, i*seq_len:(i+1)*seq_len]
            model(chunk)

    for h in hooks:
        h.remove()

    for name in layer_inputs:
        layer_inputs[name] = layer_inputs[name].cpu()

    return layer_inputs


def compute_hessian(X, damp_percent=0.01):
    H    = 2 * (X.T @ X)
    damp = damp_percent * H.diag().mean()
    H   += damp * torch.eye(H.shape[0], dtype=H.dtype)
    return H


def gptq_quantize_layer(weight, H, bits=4, group_size=128):
    out_ch, in_ch = weight.shape
    num_groups    = in_ch // group_size
    W             = weight.clone().float()
    q_min         = -(2 ** (bits - 1))
    q_max         =   2 ** (bits - 1) - 1

    try:
        L     = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)
    except torch.linalg.LinAlgError:
        damp  = 0.1 * H.diag().mean()
        H    += damp * torch.eye(H.shape[0])
        L     = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)

    W_q    = torch.zeros_like(W, dtype=torch.int8)
    scales = torch.zeros(out_ch, num_groups, dtype=torch.float32)
    W_deq  = torch.zeros_like(W)

    for col in range(in_ch):
        g = col // group_size
        if col % group_size == 0:
            group_w   = W[:, col:col + group_size]
            group_max = group_w.abs().max(dim=1).values.clamp(min=1e-8)
            scale     = group_max / q_max
            scales[:, g] = scale

        w_col     = W[:, col]
        w_q_col   = torch.clamp(torch.round(w_col / scale), q_min, q_max).to(torch.int8)
        w_deq_col = w_q_col.float() * scale

        W_q[:, col]   = w_q_col
        W_deq[:, col] = w_deq_col

        delta_w  = w_deq_col - w_col
        h_inv_qq = H_inv[col, col]
        if h_inv_qq.abs() > 1e-10:
            correction       = torch.outer(delta_w, H_inv[col, col+1:]) / h_inv_qq
            W[:, col+1:]    -= correction

    return W_q, scales, W_deq


def apply_gptq(model, encodings, bits=4, group_size=128, n_samples=512, seq_len=512):
    linear_names = [
        name for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]
    print(f"Found {len(linear_names)} Linear layers to quantize with GPTQ.")
    print(f"Bits: INT{bits}, group_size: {group_size}")

    named_modules_dict = dict(model.named_modules())

    for i, name in enumerate(linear_names):
        linear = named_modules_dict[name]
        W      = linear.weight.data.float().cpu()
        bias   = linear.bias.data if linear.bias is not None else None

        layer_inputs = collect_layer_inputs(
            model, encodings, [name], n_samples=n_samples, seq_len=seq_len
        )

        if name not in layer_inputs or layer_inputs[name].numel() == 0:
            print(f"  [{i+1}/{len(linear_names)}] Skipping {name}")
            continue

        X = layer_inputs[name]
        H = compute_hessian(X, damp_percent=0.01)

        W_q, scales, W_deq = gptq_quantize_layer(W, H, bits=bits, group_size=group_size)

        new_module = GPTQ_QuantizedLinear(W_q, scales, bias, group_size=group_size)

        parts  = name.split(".")
        parent = modelca
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_module)

        del X, H, W_q, scales, W_deq, layer_inputs
        torch.cuda.empty_cache()

        if (i + 1) % 10 == 0 or (i + 1) == len(linear_names):
            print(f"  [{i+1}/{len(linear_names)}] done")

    return model


# ── Run GPTQ ──────────────────────────────────────────────────────────────────
print("Running GPTQ (INT4, group_size=128)...")
start = time.time()
model = apply_gptq(model, encodings, bits=4, group_size=128,
                   n_samples=512, seq_len=512)
elapsed = time.time() - start
print(f"\nGPTQ done in {elapsed/60:.1f} min")
print(f"Model size: {model_size_gb(model):.2f} GB")

Running GPTQ (INT4, group_size=128)...
Found 225 Linear layers to quantize with GPTQ.
Bits: INT4, group_size: 128
  [10/225] done


KeyboardInterrupt: 

In [7]:
# ── GPTQ: All definitions + run ───────────────────────────────────────────────

class GPTQ_QuantizedLinear(nn.Module):
    def __init__(self, weight_q, scales, bias=None, group_size=128):
        super().__init__()
        self.group_size = group_size
        self.register_buffer("weight_q", weight_q)
        self.register_buffer("scales",   scales)
        self.register_buffer("bias",     bias)

    def forward(self, x):
        out_ch, in_ch   = self.weight_q.shape
        scales_expanded = self.scales.repeat_interleave(self.group_size, dim=1)
        weight_deq      = self.weight_q.float() * scales_expanded
        weight_deq      = weight_deq.to(dtype=x.dtype, device=x.device)
        return nn.functional.linear(x, weight_deq, self.bias)


def collect_layer_inputs(model, encodings, layer_names, n_samples=512, seq_len=128):
    layer_inputs = {}
    hooks        = []

    for name, module in model.named_modules():
        if name not in layer_names:
            continue
        if not isinstance(module, nn.Linear):
            continue

        def make_hook(n):
            def hook(module, input, output):
                x = input[0].detach().view(-1, input[0].shape[-1]).float()
                if n not in layer_inputs:
                    layer_inputs[n] = x
                else:
                    layer_inputs[n] = torch.cat([layer_inputs[n], x], dim=0)
            return hook

        hooks.append(module.register_forward_hook(make_hook(name)))

    input_ids = encodings.input_ids.to(device)
    n_batches = n_samples // seq_len

    with torch.no_grad():
        for i in range(n_batches):
            chunk = input_ids[:, i*seq_len:(i+1)*seq_len]
            model(chunk)

    for h in hooks:
        h.remove()

    for name in layer_inputs:
        layer_inputs[name] = layer_inputs[name].cpu()

    return layer_inputs


def compute_hessian(X, damp_percent=0.01):
    H    = 2 * (X.T @ X)
    damp = damp_percent * H.diag().mean()
    H   += damp * torch.eye(H.shape[0], dtype=H.dtype)
    return H


def gptq_quantize_layer(weight, H, bits=4, group_size=128):
    out_ch, in_ch = weight.shape
    W             = weight.clone().float()
    q_min         = -(2 ** (bits - 1))
    q_max         =   2 ** (bits - 1) - 1

    try:
        L     = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)
    except torch.linalg.LinAlgError:
        damp  = 0.1 * H.diag().mean()
        H    += damp * torch.eye(H.shape[0])
        L     = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)

    W_q    = torch.zeros_like(W, dtype=torch.int8)
    scales = torch.zeros(out_ch, in_ch // group_size, dtype=torch.float32)
    W_deq  = torch.zeros_like(W)

    for col in range(in_ch):
        g = col // group_size
        if col % group_size == 0:
            group_w   = W[:, col:col + group_size]
            group_max = group_w.abs().max(dim=1).values.clamp(min=1e-8)
            scale     = group_max / q_max
            scales[:, g] = scale

        w_col     = W[:, col]
        w_q_col   = torch.clamp(torch.round(w_col / scale), q_min, q_max).to(torch.int8)
        w_deq_col = w_q_col.float() * scale

        W_q[:, col]   = w_q_col
        W_deq[:, col] = w_deq_col

        delta_w  = w_deq_col - w_col
        h_inv_qq = H_inv[col, col]
        if h_inv_qq.abs() > 1e-10:
            W[:, col+1:] -= torch.outer(delta_w, H_inv[col, col+1:]) / h_inv_qq

    return W_q, scales, W_deq


def apply_gptq(model, encodings, bits=4, group_size=128, n_samples=512, seq_len=128):
    # ── Option 2: skip lm_head and embed_tokens ───────────────────────────────
    linear_names = [
        name for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
        and "lm_head" not in name
        and "embed"   not in name
    ]
    total = len(linear_names)
    print(f"Found {total} Linear layers to quantize with GPTQ.")
    print(f"Bits: INT{bits}, group_size: {group_size}, seq_len: {seq_len}")
    print("-" * 60)

    named_modules_dict = dict(model.named_modules())
    start_time = time.time()

    for i, name in enumerate(linear_names):
        linear = named_modules_dict[name]
        W      = linear.weight.data.float().cpu()
        bias   = linear.bias.data if linear.bias is not None else None

        layer_inputs = collect_layer_inputs(
            model, encodings, [name],
            n_samples=n_samples, seq_len=seq_len
        )

        if name not in layer_inputs or layer_inputs[name].numel() == 0:
            print(f"  [{i+1}/{total}] SKIP {name}", flush=True)
            continue

        X = layer_inputs[name]
        H = compute_hessian(X, damp_percent=0.01)
        W_q, scales, W_deq = gptq_quantize_layer(W, H, bits=bits, group_size=group_size)

        new_module = GPTQ_QuantizedLinear(W_q, scales, bias, group_size=group_size)
        parts  = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_module)

        del X, H, W_q, scales, W_deq, layer_inputs
        torch.cuda.empty_cache()

        # ── Progress bar ──────────────────────────────────────────────────────
        elapsed      = time.time() - start_time
        avg          = elapsed / (i + 1)
        remaining    = avg * (total - i - 1)
        pct          = (i + 1) / total
        bar_len      = 30
        filled       = int(bar_len * pct)
        bar          = "█" * filled + "░" * (bar_len - filled)
        print(f"  [{bar}] {i+1}/{total} | "
              f"{elapsed/60:.1f}m elapsed | "
              f"~{remaining/60:.1f}m left | "
              f"{name.split('.')[-1]}",
              flush=True)

    return model


# ── Reload fresh model ────────────────────────────────────────────────────────
del model
torch.cuda.empty_cache()

print("Reloading Llama 3 8B in BF16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="cuda"
)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

# ── Run GPTQ ──────────────────────────────────────────────────────────────────
print("Running GPTQ (INT4, group_size=128)...")
start = time.time()
model = apply_gptq(model, encodings, bits=4, group_size=128,
                   n_samples=512, seq_len=128)
elapsed = time.time() - start
print(f"\nGPTQ done in {elapsed/60:.1f} min")
print(f"Model size: {model_size_gb(model):.2f} GB")

Reloading Llama 3 8B in BF16...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

VRAM: 16.07 GB

Running GPTQ (INT4, group_size=128)...
Found 224 Linear layers to quantize with GPTQ.
Bits: INT4, group_size: 128, seq_len: 128
------------------------------------------------------------
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 1/224 | 1.5m elapsed | ~330.2m left | q_proj
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 2/224 | 1.7m elapsed | ~192.9m left | k_proj
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 3/224 | 2.0m elapsed | ~146.7m left | v_proj
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 4/224 | 3.3m elapsed | ~178.9m left | o_proj
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 5/224 | 8.9m elapsed | ~391.6m left | gate_proj


KeyboardInterrupt: 